---
short_title: "Datenvalidierung"
numbering:
    heading_1: true
    heading_2: true
    title: true
---

# Datenvalidierung

Echte Daten sind selten perfekt. Fehlende Werte, zusätzliche Spalten, falsche
Trennzeichen oder Texte, wo Zahlen erwartet werden, sind Alltag. Bevor wir Daten
auswerten, müssen wir sie **validieren** (prüfen).

In diesem Kapitel lernen wir,

- wie wir die **Spaltenzahl** jeder Zeile prüfen,
- wie wir **fehlerhafte Zeilen protokollieren**, statt sofort abzubrechen,
- wie wir mit `try`/`except` **Typkonvertierungen** absichern.

## Ein absichtlich fehlerhafter Datensatz

Um Fehler zu sehen, erzeugen wir uns eine „schmutzige“ {term}`CSV`-Datei direkt im
Speicher. Sie enthält:

- eine Zeile mit zu wenigen Feldern,
- eine Zeile mit zu vielen Feldern,
- ein Feld, das keine ganze Zahl ist.

```csv
isbn,author,year,title,sales_year,sales
9781,Orwell,1949,1984,2020,10
9782,Kafka,keine-zahl,Der Prozess,2020,5
9783,Woolf,1927
9784,Joyce,1922,Ulysses,2020,14,zusatz
9785,Morrison,1987,Beloved,2020,7
```

In [1]:
import csv
import io

schmutzige_daten = "isbn,author,year,title,sales_year,sales\n" \
    "9781,Orwell,1949,1984,2020,10\n" \
    "9782,Kafka,keine-zahl,Der Prozess,2020,5\n" \
    "9783,Woolf,1927\n" \
    "9784,Joyce,1922,Ulysses,2020,14,zusatz\n" \
    "9785,Morrison,1987,Beloved,2020,7\n"

print(schmutzige_daten)

isbn,author,year,title,sales_year,sales
9781,Orwell,1949,1984,2020,10
9782,Kafka,keine-zahl,Der Prozess,2020,5
9783,Woolf,1927
9784,Joyce,1922,Ulysses,2020,14,zusatz
9785,Morrison,1987,Beloved,2020,7



## Spaltenzahl prüfen und Fehler sammeln

Eine einfache, aber wirkungsvolle Prüfung ist die Feldanzahl. Statt beim ersten
Fehler abzubrechen, sammeln wir alle Probleme in einer Liste.

In [2]:
reader = csv.reader(io.StringIO(schmutzige_daten))
header = next(reader)

gute_zeilen = []
fehler = []

for zeilennummer, zeile in enumerate(reader, start=2):
    if len(zeile) != len(header):
        fehler.append(
            f"Zeile {zeilennummer}: {len(zeile)} statt {len(header)} Felder"
        )
    else:
        gute_zeilen.append(zeile)

print("Gültige Zeilen:", len(gute_zeilen))
print("Fehlerhafte Zeilen:", len(fehler))
for meldung in fehler:
    print(" -", meldung)

Gültige Zeilen: 3
Fehlerhafte Zeilen: 2
 - Zeile 4: 3 statt 6 Felder
 - Zeile 5: 7 statt 6 Felder


## Typkonvertierung mit `try`/`except`

CSV liefert **alle** Werte als Strings. Zahlen müssen wir ausdrücklich umwandeln:
`int("1949")` ergibt `1949`. Steht im Feld kein gültiger Wert, löst `int()` einen
`ValueError` aus. Diesen fangen wir mit `try`/`except` ab.

In [3]:
for wert in ["1949", "keine-zahl", "19,49", ""]:
    try:
        zahl = int(wert)
        print(f"{wert!r:>12} -> {zahl}")
    except ValueError:
        print(f"{wert!r:>12} -> keine ganze Zahl")

      '1949' -> 1949
'keine-zahl' -> keine ganze Zahl
     '19,49' -> keine ganze Zahl
          '' -> keine ganze Zahl


Für **Kommazahlen** nutzen wir `float()`. Ein deutsches Komma müssen wir vorher
durch einen Punkt ersetzen.

In [4]:
preise = ["12.50", "9,99", "keine Angabe", ""]

for wert in preise:
    try:
        zahl = float(wert.replace(",", "."))
        print(f"{wert!r:>16} -> {zahl}")
    except ValueError:
        print(f"{wert!r:>16} -> keine gültige Zahl")

         '12.50' -> 12.5
          '9,99' -> 9.99
  'keine Angabe' -> keine gültige Zahl
              '' -> keine gültige Zahl


## Eine wiederverwendbare Prüffunktion

Wir bauen die Prüfungen in eine Funktion ein. Sie bekommt eine Zeile (als Liste)
und den Header und gibt eine Liste von Fehlermeldungen zurück. Eine leere Liste
bedeutet: alles in Ordnung.

In [5]:
def pruefe_zeile(zeile, spalten):
    """Prüft eine CSV-Zeile und gibt eine Liste mit Fehlermeldungen zurück."""
    meldungen = []

    # 1. Stimmt die Anzahl der Felder?
    if len(zeile) != len(spalten):
        meldungen.append(f"{len(zeile)} statt {len(spalten)} Felder")
        # Ohne passende Spalten ist keine weitere Prüfung sinnvoll.
        return meldungen

    # 2. Lassen sich die Zahlenspalten in int umwandeln?
    for name in ("year", "sales_year", "sales"):
        index = spalten.index(name)
        try:
            int(zeile[index])
        except ValueError:
            meldungen.append(f"{name}={zeile[index]!r} ist keine ganze Zahl")

    return meldungen

In [6]:
reader = csv.reader(io.StringIO(schmutzige_daten))
header = next(reader)

for zeilennummer, zeile in enumerate(reader, start=2):
    meldungen = pruefe_zeile(zeile, header)
    status = "OK" if not meldungen else "FEHLER"
    print(f"Zeile {zeilennummer}: {status:6} | {zeile}")
    for meldung in meldungen:
        print(f"         -> {meldung}")

Zeile 2: OK     | ['9781', 'Orwell', '1949', '1984', '2020', '10']
Zeile 3: FEHLER | ['9782', 'Kafka', 'keine-zahl', 'Der Prozess', '2020', '5']
         -> year='keine-zahl' ist keine ganze Zahl
Zeile 4: FEHLER | ['9783', 'Woolf', '1927']
         -> 3 statt 6 Felder
Zeile 5: FEHLER | ['9784', 'Joyce', '1922', 'Ulysses', '2020', '14', 'zusatz']
         -> 7 statt 6 Felder
Zeile 6: OK     | ['9785', 'Morrison', '1987', 'Beloved', '2020', '7']


## Gute und schlechte Datensätze nebeneinander

Wir trennen die Datensätze in zwei Listen. So können wir die gültigen Daten
weiterverarbeiten und die fehlerhaften zur Kontrolle ausgeben.

In [7]:
reader = csv.reader(io.StringIO(schmutzige_daten))
header = next(reader)

gute_daten = []
schlechte_daten = []

for zeilennummer, zeile in enumerate(reader, start=2):
    meldungen = pruefe_zeile(zeile, header)
    if meldungen:
        schlechte_daten.append((zeilennummer, zeile, meldungen))
    else:
        gute_daten.append(dict(zip(header, zeile)))

print("=== Gültige Datensätze ===")
for datensatz in gute_daten:
    print(datensatz)

print()
print("=== Verworfene Datensätze ===")
for zeilennummer, zeile, meldungen in schlechte_daten:
    print(f"Zeile {zeilennummer}: {'; '.join(meldungen)}")

=== Gültige Datensätze ===
{'isbn': '9781', 'author': 'Orwell', 'year': '1949', 'title': '1984', 'sales_year': '2020', 'sales': '10'}
{'isbn': '9785', 'author': 'Morrison', 'year': '1987', 'title': 'Beloved', 'sales_year': '2020', 'sales': '7'}

=== Verworfene Datensätze ===
Zeile 3: year='keine-zahl' ist keine ganze Zahl
Zeile 4: 3 statt 6 Felder
Zeile 5: 7 statt 6 Felder


## Validierung des echten Datensatzes

Ein sauber erzeugter Datensatz sollte alle Prüfungen bestehen. Prüfen wir das
für `books_powerlaw_dataset.csv`.

In [8]:
datei = "../assets/data/books_powerlaw_dataset.csv"

with open(datei, encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    anzahl_ok = 0
    anzahl_fehler = 0

    for zeilennummer, zeile in enumerate(reader, start=2):
        if pruefe_zeile(zeile, header):
            anzahl_fehler += 1
            print(f"Zeile {zeilennummer} fehlerhaft:", zeile)
        else:
            anzahl_ok += 1

print(f"Gültige Zeilen: {anzahl_ok}, fehlerhafte Zeilen: {anzahl_fehler}")

Gültige Zeilen: 180, fehlerhafte Zeilen: 0


## Zusammenfassung

- Prüfen Sie zuerst die **Spaltenzahl**: Sie deckt viele strukturelle Fehler auf.
- Sammeln Sie Fehler in einer Liste, statt beim ersten Problem abzubrechen.
- CSV-Werte sind Strings. Nutzen Sie `try`/`except` um `int()` und `float()`.
- Trennen Sie gültige und ungültige Datensätze und protokollieren Sie die
  verworfenen Zeilen mit Zeilennummer.

```{hint}
Im nächsten Kapitel übergeben wir den Pfad zur CSV-Datei als
Kommandozeilen-Parameter, damit wir das Skript ohne Code-Änderung auf andere
Dateien anwenden können.
```